In [1]:
import pandas as pd 
data=pd.read_csv("./data/data.csv")

In [2]:
data.head()

,Authors,Article Title,Article Link
0,"Lam CSP, Bozkurt B, Cherney DZI, Ezekowitz JA,...",Kidney disease and heart failure: recent advan...,https://europepmc.org/article/MED/41791738
1,"Stubbendorff A, Ericson U, Hallström E, Samuel...",Nutritional adequacy of the EAT-Lancet diet: a...,https://europepmc.org/article/MED/41692025
2,"Wolf J, Goggin KP, Inaba Y, Allison KJ, Ahmed ...",Predicting bloodstream infection by plasma cel...,https://europepmc.org/article/MED/41780551
3,"Kennedy OJ, Chauhan U, Gorman L, Lorigan P, Me...",Prostate Cancer Care for Men with an Intellect...,https://europepmc.org/article/MED/41720694
4,Ingrisch M.,The architectural gap in clinical artificial i...,https://europepmc.org/article/MED/41786547


In [3]:
import pandas as pd
import requests
from bs4 import BeautifulSoup
from concurrent.futures import ThreadPoolExecutor, as_completed
import json
import threading

# Assume your dataset is already in a variable called 'data'
if 'Abstract' not in data.columns:
    data['Abstract'] = ""

SAVE_FILE = "data_with_abstracts.json"
SAVE_EVERY = 5  # save after this many rows
HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                  "AppleWebKit/537.36 (KHTML, like Gecko) "
                  "Chrome/112.0.0.0 Safari/537.36"
}

# Lock for thread-safe operations
lock = threading.Lock()

from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
import time
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC


def fetch_abstract(idx, url, title):
    """Fetch abstract from Europe PMC article page using Selenium."""

    driver = None
    try:
        options = Options()
        options.add_argument("--headless=new")   # better headless mode
        options.add_argument("--no-sandbox")
        options.add_argument("--window-size=1920,1080")

        driver = webdriver.Chrome(options=options)

        driver.get(url)

        wait = WebDriverWait(driver, 10)

        # Wait for abstract container
        abstract_div = wait.until(
            EC.presence_of_element_located(
                (By.CSS_SELECTOR, "#abstract")
            )
        )

        abstract_text = abstract_div.text.strip()

        print(f"[{idx}] Processed: {title}")

    except Exception as e:
        abstract_text = f"Error fetching abstract: {e}"
        print(f"[{idx}] Error processing {title}: {e}")

    finally:
        if driver:
            driver.quit()

    return idx, abstract_text


def save_json_periodically(data, save_file):
    """Save the DataFrame to JSON."""
    with lock:
        data.to_json(save_file, orient='records', force_ascii=False, indent=2)
        print(f"Progress saved to {save_file}")



In [ ]:
# Prepare tasks for rows that don't have an abstract yet
tasks = [(idx, row['Article Link'], row['Article Title']) 
         for idx, row in data.iterrows() if not row['Abstract'] and idx>442]

# Use ThreadPoolExecutor for parallel scraping
MAX_WORKERS = 25  # adjust based on your connection
completed = 0

with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    future_to_idx = {executor.submit(fetch_abstract, idx, url, title): idx for idx, url, title in tasks}

    for future in as_completed(future_to_idx):
        idx, abstract = future.result()
        data.at[idx, 'Abstract'] = abstract
        completed += 1

        # Periodic save
        if completed % SAVE_EVERY == 0:
            save_json_periodically(data, SAVE_FILE)

# Final save
save_json_periodically(data, SAVE_FILE)
print("All done! Dataset saved to JSON.")

[448] Processed: Rice starch-ferulic acid complexes: The role of extrusion process
[460] Processed: [THE POTENTIAL OF STRATEGIC TECHNOLOGICAL PARTNERSHIP IN ORGANIZATION OF SOCIAL DEMOGRAPHIC STUDIES IN THE FIELD OF DEVELOPMENT OF RURAL TERRITORIES].
[451] Processed: Nutritional adequacy of the EAT-Lancet diet: a Swedish population-based cohort study.
[449] Processed: Effect of barium sulfate, thickener type, and saline solution on the rheological properties of liquids used for instrumental swallowing assessment
[445] Processed: Enhanced sesamol via infrared roasting-grinding: Molecular activation
[446] Processed: A Bayesian Gaussian Process Regression soft sensor for industrial sugar crystallization monitoring
[447] Processed: Electrosprayed hempseed protein/oilbody emulsions for encapsulation and oxidative stabilization of PUFA-rich oils
[452] Processed: Predicting bloodstream infection by plasma cell-free metagenomic sequencing: a prospective cohort study.
[458] Processed: Update of